# Layer 1b — User 行為異常偵測

目標：找出 user 行為導致的異常。
- 同一 device 短時間內多筆訂單（contention）
- file 數量異常大的訂單

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv('../data/orders.csv')
df['order_created_at'] = pd.to_datetime(df['order_created_at'], format='%Y/%m/%d %I:%M:%S %p')
print(f"Total orders: {len(df)}")


Total orders: 30000


## 1. Contention 偵測

同一 device 在 30 分鐘內有多筆訂單，可能造成 resource contention。

In [2]:
# Sort by device and time, then detect contention windows
df = df.sort_values(['device_id', 'order_created_at'])

# For each order, count how many orders on same device within +/- 30 min
WINDOW_MINUTES = 30

def count_contention(group):
    times = group['order_created_at'].values
    counts = []
    for i, t in enumerate(times):
        window_start = t - np.timedelta64(WINDOW_MINUTES, 'm')
        window_end = t + np.timedelta64(WINDOW_MINUTES, 'm')
        count = ((times >= window_start) & (times <= window_end)).sum()
        counts.append(count)
    return pd.Series(counts, index=group.index)

df['contention_count'] = df.groupby('device_id', group_keys=False).apply(count_contention)

# Flag contention: 3+ orders on same device within window
df['is_contention'] = df['contention_count'] >= 3
contention_orders = df[df['is_contention']]
print(f"Contention orders (≥3 within {WINDOW_MINUTES}min window): {len(contention_orders)} ({100*len(contention_orders)/len(df):.1f}%)")
print(f"Devices with contention: {contention_orders['device_id'].nunique()}")


Contention orders (≥3 within 30min window): 584 (1.9%)
Devices with contention: 148


## 2. 異常大 file_count 偵測

In [3]:
# Flag orders with abnormally large file_count (global Q3 + 3*IQR)
q1 = df['file_count'].quantile(0.25)
q3 = df['file_count'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 3 * iqr
df['is_large_filecount'] = df['file_count'] > upper
large_fc = df[df['is_large_filecount']]
print(f"Large file_count threshold: > {upper:.0f}")
print(f"Large file_count orders: {len(large_fc)} ({100*len(large_fc)/len(df):.1f}%)")


Large file_count threshold: > 2727
Large file_count orders: 900 (3.0%)


## 3. 綜合 User 行為異常標記

In [4]:
df['is_user_anomaly'] = df['is_contention'] | df['is_large_filecount']
user_anomalies = df[df['is_user_anomaly']]
print(f"Total user anomalies: {len(user_anomalies)} ({100*len(user_anomalies)/len(df):.1f}%)")
print(f"  - Contention only: {(df['is_contention'] & ~df['is_large_filecount']).sum()}")
print(f"  - Large file_count only: {(~df['is_contention'] & df['is_large_filecount']).sum()}")
print(f"  - Both: {(df['is_contention'] & df['is_large_filecount']).sum()}")


Total user anomalies: 1459 (4.9%)
  - Contention only: 559
  - Large file_count only: 875
  - Both: 25


## 4. 圖表

In [5]:
# Chart 1: Contention heatmap - orders per device per hour
contention_df = df[df['is_contention']].copy()
contention_df['hour'] = contention_df['order_created_at'].dt.hour
contention_df['date'] = contention_df['order_created_at'].dt.date

# Top 15 devices by contention count
top_contention_devices = contention_df['device_id'].value_counts().head(15).index
heat_data = contention_df[contention_df['device_id'].isin(top_contention_devices)]
heat_pivot = heat_data.groupby(['device_id', 'hour']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(heat_pivot, cmap='YlOrRd', annot=True, fmt='d', ax=ax, linewidths=0.5)
ax.set_title('Order Contention Heatmap: Orders per Hour (Top 15 Contention Devices)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Device ID')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '1b_contention_heatmap.png', dpi=150)
plt.show()
print("Saved: 1b_contention_heatmap.png")


Saved: 1b_contention_heatmap.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65487/89475457.py:18: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [6]:
# Chart 2: File count distribution with anomaly threshold
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(df['file_count'], bins=100, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(x=upper, color='red', linestyle='--', linewidth=2, label=f'Threshold: {upper:.0f}')
ax.set_title('File Count Distribution with Anomaly Threshold')
ax.set_xlabel('File Count')
ax.set_ylabel('Number of Orders')
ax.legend()
ax.set_xlim(0, df['file_count'].quantile(0.99) * 1.2)
plt.tight_layout()
plt.savefig(REPORTS_DIR / '1b_filecount_distribution.png', dpi=150)
plt.show()
print("Saved: 1b_filecount_distribution.png")


Saved: 1b_filecount_distribution.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65487/1087838851.py:12: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [7]:
# Chart 3: Contention impact on duration
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: contention vs no contention duration
for label, subset, color in [('No Contention', df[~df['is_contention']], 'steelblue'),
                               ('Contention', df[df['is_contention']], 'coral')]:
    axes[0].hist(subset['total_duration_seconds'].clip(upper=5000), bins=80, alpha=0.6,
                 label=label, color=color, density=True)
axes[0].set_title('Duration Distribution: Contention vs Normal')
axes[0].set_xlabel('Total Duration (seconds, clipped at 5000)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Right: scatter of contention_count vs duration
ax = axes[1]
scatter = ax.scatter(df['contention_count'], df['total_duration_seconds'],
                     alpha=0.1, s=5, c=df['file_count'], cmap='viridis')
ax.set_title('Contention Count vs Total Duration')
ax.set_xlabel('Concurrent Orders on Same Device')
ax.set_ylabel('Total Duration (seconds)')
ax.set_yscale('log')
plt.colorbar(scatter, ax=ax, label='File Count')

plt.tight_layout()
plt.savefig(REPORTS_DIR / '1b_contention_impact.png', dpi=150)
plt.show()
print("Saved: 1b_contention_impact.png")


Saved: 1b_contention_impact.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65487/603273922.py:26: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 5. 匯出標記

In [8]:
# Export user anomaly flags
user_flags = df[['order_id', 'is_user_anomaly', 'is_contention', 'is_large_filecount']].copy()
user_flags.to_csv('../data/user_anomaly_flags.csv', index=False)
print(f"Exported {user_flags['is_user_anomaly'].sum()} user anomaly flags")

print(f"\n=== Layer 1b Summary ===")
print(f"Total orders: {len(df)}")
print(f"User anomalies: {len(user_anomalies)} ({100*len(user_anomalies)/len(df):.1f}%)")
print(f"Contention events: {df['is_contention'].sum()}")
print(f"Large file_count: {df['is_large_filecount'].sum()}")


Exported 1459 user anomaly flags

=== Layer 1b Summary ===
Total orders: 30000
User anomalies: 1459 (4.9%)
Contention events: 584
Large file_count: 900
